Given a DataFrame with columns emp_id, manager_id, salary, find employees whose salary is greater than their manager's salary

In [0]:
from pyspark.sql import Row

# Sample dataset creation
data = [
    Row(emp_id=1, manager_id=3, salary=5000),
    Row(emp_id=2, manager_id=3, salary=7000),
    Row(emp_id=3, manager_id=None, salary=8000),
    Row(emp_id=4, manager_id=2, salary=9000),
    Row(emp_id=5, manager_id=1, salary=4000)
]

df = spark.createDataFrame(data)
display(df)

this inner join approach is good. because there is no need to use collect. so no driver involved.


In [0]:
from pyspark.sql.functions import col

# Self join to compare employee salary with manager salary
df_with_mgr_salary = df.alias("emp").join(
    df.alias("mgr"),
    col("emp.manager_id") == col("mgr.emp_id"),
    how="inner"
).select(
    col("emp.emp_id"),
    col("emp.manager_id"),
    col("emp.salary"),
    col("mgr.salary").alias("manager_salary")
).where(
    col("emp.salary") > col("mgr.salary")
)

display(df_with_mgr_salary)

my solution is below


In [0]:
manager_id_salary = df.where(col("manager_id").isNull()).collect()[0]
# manager_id_salary['salary']

highest_sal = df.where(col('salary')> manager_id_salary['salary'])
highest_sal.show()

another approach
this i felt very lengthy approach


In [0]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import LongType

# Create a mapping of emp_id to salary for managers
manager_salary_map = {row.emp_id: row.salary for row in df.select("emp_id", "salary").collect()}

# Create UDF to lookup manager salary
def get_manager_salary(manager_id):
    if manager_id is None:
        return None
    return manager_salary_map.get(manager_id)

get_manager_salary_udf = udf(get_manager_salary, LongType())

# Add manager_salary column using UDF
df_with_mgr_salary = df.withColumn(
    "manager_salary",
    get_manager_salary_udf(col("manager_id"))
)

df_with_mgr_salary.show()

# Filter employees whose salary is greater than their manager's salary
result = df_with_mgr_salary.where(col("salary") > col("manager_salary"))

display(result)